# Chicago Crime Dataset – Data Cleaning and Preprocessing

## Project Overview
This notebook performs data cleaning and preprocessing on the Chicago
Police Department crime dataset.

## Objectives
1. Load the raw crime dataset.
2. Inspect and repair malformed CSV records.
3. Identify and handle missing values.
4. Remove duplicate records.
5. Convert date columns into datetime format.
6. Create Year, Month and DayOfWeek features.
7. Standardize categorical variables.
8. Map IUCR codes to primary crime categories.
9. Validate the cleaned dataset.
10. Export the cleaned dataset for SQLite3 integration.

In [1]:
import pandas as pd
import numpy as np
import io

from pathlib import Path

# Project directory
BASE_DIR = Path("..").resolve()

# Raw data directory
RAW_DIR = BASE_DIR / "data" / "raw"

# Processed data directory
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Create processed directory if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Input files
CRIME_FILE = RAW_DIR / "chicago_crime_dataset.csv"
IUCR_FILE = RAW_DIR / "iucr_codes.csv"

# Output file
OUTPUT_FILE = PROCESSED_DIR / "chicago_crime_cleaned.csv"

print("Project directory:", BASE_DIR)
print("Crime dataset exists:", CRIME_FILE.exists())
print("IUCR reference exists:", IUCR_FILE.exists())
print("Processed directory:", PROCESSED_DIR)

Project directory: C:\Users\deepe\Downloads\chicago_crime_capstone
Crime dataset exists: True
IUCR reference exists: True
Processed directory: C:\Users\deepe\Downloads\chicago_crime_capstone\data\processed


In [2]:
def read_chicago_crime_csv(path):

    # Read original CSV text
    text = path.read_text(
        encoding="utf-8-sig",
        errors="replace"
    )

    fixed_lines = []
    repaired_line_numbers = []

    # Repair lines containing unmatched quotation marks
    for line_number, line in enumerate(
        text.splitlines(),
        start=1
    ):

        if line.count('"') % 2 == 1:
            line = line + '"'
            repaired_line_numbers.append(line_number)

        fixed_lines.append(line)

    # Reconstruct CSV content
    csv_text = "\n".join(fixed_lines)

    # Parse CSV
    dataframe = pd.read_csv(
        io.StringIO(csv_text),
        low_memory=False
    )

    # Validate expected structure
    if dataframe.shape[1] != 20:
        raise ValueError(
            f"Expected 20 columns, got {dataframe.shape[1]}"
        )

    return dataframe, repaired_line_numbers


# Load crime dataset
crime_df, repaired_lines = read_chicago_crime_csv(
    CRIME_FILE
)

print("Raw dataset loaded successfully.")
print("Dataset shape:", crime_df.shape)
print("Repaired line numbers:", repaired_lines)

Raw dataset loaded successfully.
Dataset shape: (498, 20)
Repaired line numbers: [498, 499]


In [3]:
# Load IUCR reference data
iucr_df = pd.read_csv(
    IUCR_FILE,
    dtype={"iucr": "string"}
)

print("IUCR reference dataset loaded.")
print("IUCR dataset shape:", iucr_df.shape)

display(iucr_df.head())

IUCR reference dataset loaded.
IUCR dataset shape: (410, 4)


,IUCR,PRIMARY_TYPE,DESCRIPTION,INDEX CODE
0,110,HOMICIDE,FIRST DEGREE MURDER,I
1,130,HOMICIDE,SECOND DEGREE MURDER,I
2,141,HOMICIDE,INVOLUNTARY MANSLAUGHTER,N
3,142,HOMICIDE,RECKLESS HOMICIDE,N
4,261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,I


## 1. Raw Dataset Inspection

In [4]:
print("Dataset shape:")
print(crime_df.shape)

print("\nColumn names:")
print(crime_df.columns.tolist())

print("\nFirst 10 records:")
display(crime_df.head(10))

print("\nDataset information:")
crime_df.info()

Dataset shape:
(498, 20)

Column names:
['id', 'case_number', 'date', 'block', 'iucr_code', 'location_desc', 'arrest', 'domestic', 'beat_num', 'district_code', 'ward_no', 'community_code', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'date_of_update', 'latitude', 'longitude', 'location']

First 10 records:


,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location
0,10508693,HZ250496,5/3/2016,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,24,29,08B,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,10508695,HZ250409,5/3/2016,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,20,42,08B,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,10508697,HZ250503,5/3/2016,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,37,25,24,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,10508698,HZ250424,5/3/2016,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,28,25,08B,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,10508699,HZ250455,5/3/2016,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,28,25,6,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)"
5,10508702,HZ250447,5/3/2016,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,8,44,04B,1183336.0,1850642.0,2016,5/10/2016,41.745354,-87.603799,"(41.745354023, -87.603798903)"
6,10508703,HZ250489,5/3/2016,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,3,35,08B,1176730.0,1886544.0,2016,5/10/2016,41.844024,-87.626923,"(41.844023772, -87.626923253)"
7,10508704,HZ250514,5/3/2016,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,3,38,08B,1178514.0,1874573.0,2016,5/10/2016,41.811134,-87.620741,"(41.811133958, -87.62074077)"
8,10508709,HZ250523,5/3/2016,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,40,1,08B,1165696.0,1942616.0,2016,5/10/2016,41.998131,-87.665814,"(41.99813061, -87.665814038)"
9,10508982,HZ250667,5/3/2016,069XX S ASHLAND AVE,486,STREET,False,True,735,7,17,67,08B,1166876.0,1858796.0,2016,5/10/2016,41.768097,-87.663879,"(41.768096835, -87.663878589)"



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              498 non-null    int64  
 1   case_number     498 non-null    str    
 2   date            498 non-null    str    
 3   block           498 non-null    str    
 4   iucr_code       498 non-null    str    
 5   location_desc   494 non-null    str    
 6   arrest          498 non-null    bool   
 7   domestic        498 non-null    bool   
 8   beat_num        498 non-null    int64  
 9   district_code   498 non-null    int64  
 10  ward_no         498 non-null    int64  
 11  community_code  498 non-null    int64  
 12  fbi_code        498 non-null    str    
 13  x_coordinate    469 non-null    float64
 14  y_coordinate    469 non-null    float64
 15  year            498 non-null    int64  
 16  date_of_update  498 non-null    str    
 17  latitude        469 non-

In [5]:
# Standardize crime dataset column names
crime_df.columns = (
    crime_df.columns
    .str.strip()
    .str.lower()
)

# Standardize IUCR reference column names
iucr_df.columns = (
    iucr_df.columns
    .str.strip()
    .str.lower()
)

print("Crime columns:")
print(crime_df.columns.tolist())

print("\nIUCR columns:")
print(iucr_df.columns.tolist())

Crime columns:
['id', 'case_number', 'date', 'block', 'iucr_code', 'location_desc', 'arrest', 'domestic', 'beat_num', 'district_code', 'ward_no', 'community_code', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'date_of_update', 'latitude', 'longitude', 'location']

IUCR columns:
['iucr', 'primary_type', 'description', 'index code']


## 2. Missing Value Analysis

Identify missing values and calculate their percentages.
Columns with more than 50% missing values will be reviewed for possible removal.

In [6]:
# Calculate missing values
missing_summary = pd.DataFrame({
    "missing_count": crime_df.isnull().sum(),
    "missing_percentage": (
        crime_df.isnull().sum() / len(crime_df) * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percentage
location,29,5.82
y_coordinate,29,5.82
longitude,29,5.82
latitude,29,5.82
x_coordinate,29,5.82
location_desc,4,0.80
case_number,0,0.00
id,0,0.00
block,0,0.00
date,0,0.00


In [7]:
# Fill missing location descriptions
crime_df["location_desc"] = (
    crime_df["location_desc"]
    .fillna("UNKNOWN")
)

# Check missing values after handling
print("Missing values after handling:")

display(
    crime_df.isnull().sum()
    .sort_values(ascending=False)
)

print("\nDataset shape:", crime_df.shape)

Missing values after handling:


location          29
y_coordinate      29
longitude         29
latitude          29
x_coordinate      29
date               0
case_number        0
id                 0
block              0
iucr_code          0
arrest             0
location_desc      0
community_code     0
ward_no            0
district_code      0
beat_num           0
domestic           0
fbi_code           0
year               0
date_of_update     0
dtype: int64


Dataset shape: (498, 20)


## 3. Duplicate Record Detection and Removal

In [8]:
# Count exact duplicate rows
duplicate_rows = crime_df.duplicated().sum()

# Count duplicate crime IDs
duplicate_ids = crime_df["id"].duplicated().sum()

print("Exact duplicate rows:", duplicate_rows)
print("Duplicate crime IDs:", duplicate_ids)

# Remove exact duplicate rows
crime_df = crime_df.drop_duplicates()

# Keep the first record for each crime ID, if duplicates exist
crime_df = crime_df.drop_duplicates(
    subset=["id"],
    keep="first"
)

print("\nDataset shape after duplicate removal:")
print(crime_df.shape)

Exact duplicate rows: 0
Duplicate crime IDs: 0

Dataset shape after duplicate removal:
(498, 20)


## 4. Date Conversion and Feature Engineering

Convert date fields into datetime format and derive Year, Month and DayOfWeek.

In [9]:
# Convert date columns to datetime
crime_df["date"] = pd.to_datetime(
    crime_df["date"],
    errors="coerce"
)

crime_df["date_of_update"] = pd.to_datetime(
    crime_df["date_of_update"],
    errors="coerce"
)

# Check invalid dates
print("Invalid crime dates:", crime_df["date"].isna().sum())
print("Invalid update dates:", crime_df["date_of_update"].isna().sum())

# Stop if any crime dates could not be parsed
if crime_df["date"].isna().any():
    raise ValueError("Some crime dates could not be converted.")

# Create date-derived columns
crime_df["Year"] = crime_df["date"].dt.year
crime_df["Month"] = crime_df["date"].dt.month
crime_df["DayOfWeek"] = (
    crime_df["date"].dt.day_name().str.upper()
)

display(
    crime_df[
        ["date", "Year", "Month", "DayOfWeek"]
    ].head(10)
)

print("\nDate conversion completed.")

Invalid crime dates: 0
Invalid update dates: 0


,date,Year,Month,DayOfWeek
0,2016-05-03,2016,5,TUESDAY
1,2016-05-03,2016,5,TUESDAY
2,2016-05-03,2016,5,TUESDAY
3,2016-05-03,2016,5,TUESDAY
4,2016-05-03,2016,5,TUESDAY
5,2016-05-03,2016,5,TUESDAY
6,2016-05-03,2016,5,TUESDAY
7,2016-05-03,2016,5,TUESDAY
8,2016-05-03,2016,5,TUESDAY
9,2016-05-03,2016,5,TUESDAY



Date conversion completed.


## 5. Categorical Data Standardization

Remove unnecessary whitespace and standardize text values to uppercase.

In [10]:
# Standardize text columns
text_columns = [
    "case_number",
    "block",
    "iucr_code",
    "location_desc",
    "fbi_code"
]

for column in text_columns:
    crime_df[column] = (
        crime_df[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Standardize missing location descriptions
crime_df["location_desc"] = (
    crime_df["location_desc"]
    .fillna("UNKNOWN")
)

print("Categorical values standardized.")

print("\nSample categorical values:")
display(
    crime_df[
        ["iucr_code", "location_desc", "fbi_code"]
    ].head(10)
)

Categorical values standardized.

Sample categorical values:


,iucr_code,location_desc,fbi_code
0,486,APARTMENT,08B
1,486,RESIDENCE,08B
2,470,STREET,24
3,460,SIDEWALK,08B
4,820,RESIDENCE,6
5,041A,STREET,04B
6,460,CHA HALLWAY/STAIRWELL/ELEVATOR,08B
7,460,RESIDENCE PORCH/HALLWAY,08B
8,460,SIDEWALK,08B
9,486,STREET,08B


## 6. IUCR Code Mapping

Merge the crime dataset with the IUCR reference dataset to obtain
the primary crime category and description.

In [11]:
# Prepare IUCR reference columns
iucr_mapping = iucr_df[
    ["iucr", "primary_type", "description"]
].copy()

# Standardize IUCR codes
iucr_mapping["iucr"] = (
    iucr_mapping["iucr"]
    .astype("string")
    .str.strip()
    .str.upper()
)

iucr_mapping["primary_type"] = (
    iucr_mapping["primary_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

iucr_mapping["description"] = (
    iucr_mapping["description"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Check duplicate IUCR reference codes
print(
    "Duplicate IUCR reference codes:",
    iucr_mapping["iucr"].duplicated().sum()
)

# Remove duplicate reference codes if present
iucr_mapping = iucr_mapping.drop_duplicates(
    subset=["iucr"],
    keep="first"
)

# Merge crime records with IUCR reference
crime_df = crime_df.merge(
    iucr_mapping,
    left_on="iucr_code",
    right_on="iucr",
    how="left",
    validate="many_to_one"
)

# Remove redundant IUCR reference column
crime_df = crime_df.drop(columns=["iucr"])

# Check unmatched codes
unmatched_codes = crime_df[
    crime_df["primary_type"].isna()
]["iucr_code"].unique()

print("\nUnmatched IUCR codes:", len(unmatched_codes))
print(unmatched_codes)

print("\nPrimary crime category counts:")
display(
    crime_df["primary_type"].value_counts()
)

Duplicate IUCR reference codes: 0

Unmatched IUCR codes: 0
<ArrowStringArray>
[]
Length: 0, dtype: string

Primary crime category counts:


primary_type
THEFT                               105
BATTERY                              85
CRIMINAL DAMAGE                      49
OTHER OFFENSE                        39
DECEPTIVE PRACTICE                   38
ASSAULT                              35
NARCOTICS                            32
ROBBERY                              30
CRIMINAL TRESPASS                    17
MOTOR VEHICLE THEFT                  16
BURGLARY                             14
WEAPONS VIOLATION                     8
PUBLIC PEACE VIOLATION                5
OFFENSE INVOLVING CHILDREN            5
CRIMINAL SEXUAL ASSAULT               4
SEX OFFENSE                           3
HOMICIDE                              3
PROSTITUTION                          3
KIDNAPPING                            2
INTERFERENCE WITH PUBLIC OFFICER      2
STALKING                              1
OBSCENITY                             1
GAMBLING                              1
Name: count, dtype: int64[pyarrow]

## 7. Final Data Validation

Verify record counts, missing values, duplicate records and
the successful creation of crime categories.

In [13]:
print("Final dataset shape:", crime_df.shape)

print(
    "Duplicate rows:",
    crime_df.duplicated().sum()
)

print(
    "Duplicate crime IDs:",
    crime_df["id"].duplicated().sum()
)

print(
    "Missing primary_type values:",
    crime_df["primary_type"].isna().sum()
)

print("\nFinal missing-value summary:")

display(
    crime_df.isnull().sum()
    .sort_values(ascending=False)
)

print("\nFinal dataset preview:")
display(crime_df.head(10))

Final dataset shape: (498, 25)
Duplicate rows: 0
Duplicate crime IDs: 0
Missing primary_type values: 0

Final missing-value summary:


y_coordinate      29
latitude          29
longitude         29
x_coordinate      29
location          29
case_number        0
date               0
block              0
id                 0
beat_num           0
domestic           0
arrest             0
location_desc      0
iucr_code          0
fbi_code           0
district_code      0
ward_no            0
date_of_update     0
year               0
community_code     0
Year               0
Month              0
DayOfWeek          0
primary_type       0
description        0
dtype: int64


Final dataset preview:


,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,...,year,date_of_update,latitude,longitude,location,Year,Month,DayOfWeek,primary_type,description
0,10508693,HZ250496,2016-05-03,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,...,2016,2016-05-10,41.864073,-87.706819,"(41.864073157, -87.706818608)",2016,5,TUESDAY,BATTERY,DOMESTIC BATTERY SIMPLE
1,10508695,HZ250409,2016-05-03,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,...,2016,2016-05-10,41.782922,-87.604363,"(41.782921527, -87.60436317)",2016,5,TUESDAY,BATTERY,DOMESTIC BATTERY SIMPLE
2,10508697,HZ250503,2016-05-03,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,...,2016,2016-05-10,41.894908,-87.758372,"(41.894908283, -87.758371958)",2016,5,TUESDAY,PUBLIC PEACE VIOLATION,RECKLESS CONDUCT
3,10508698,HZ250424,2016-05-03,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,...,2016,2016-05-10,41.885687,-87.749516,"(41.885686845, -87.749515983)",2016,5,TUESDAY,BATTERY,SIMPLE
4,10508699,HZ250455,2016-05-03,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,...,2016,2016-05-10,41.886297,-87.761751,"(41.886297242, -87.761750709)",2016,5,TUESDAY,THEFT,$500 AND UNDER
5,10508702,HZ250447,2016-05-03,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,...,2016,2016-05-10,41.745354,-87.603799,"(41.745354023, -87.603798903)",2016,5,TUESDAY,BATTERY,AGGRAVATED - HANDGUN
6,10508703,HZ250489,2016-05-03,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,...,2016,2016-05-10,41.844024,-87.626923,"(41.844023772, -87.626923253)",2016,5,TUESDAY,BATTERY,SIMPLE
7,10508704,HZ250514,2016-05-03,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,...,2016,2016-05-10,41.811134,-87.620741,"(41.811133958, -87.62074077)",2016,5,TUESDAY,BATTERY,SIMPLE
8,10508709,HZ250523,2016-05-03,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,...,2016,2016-05-10,41.998131,-87.665814,"(41.99813061, -87.665814038)",2016,5,TUESDAY,BATTERY,SIMPLE
9,10508982,HZ250667,2016-05-03,069XX S ASHLAND AVE,486,STREET,False,True,735,7,...,2016,2016-05-10,41.768097,-87.663879,"(41.768096835, -87.663878589)",2016,5,TUESDAY,BATTERY,DOMESTIC BATTERY SIMPLE


## 8. Export Cleaned Dataset

Save the cleaned dataset as a CSV file for SQLite3 database integration.

In [14]:
# Export cleaned dataset
crime_df.to_csv(
    OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)

print("Cleaned dataset exported successfully.")
print("Output file:", OUTPUT_FILE)
print("Records exported:", len(crime_df))
print("Columns exported:", len(crime_df.columns))

Cleaned dataset exported successfully.
Output file: C:\Users\deepe\Downloads\chicago_crime_capstone\data\processed\chicago_crime_cleaned.csv
Records exported: 498
Columns exported: 25


In [15]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"

dimension_files = {
    "District": "chicago_district_ps_info.csv",
    "Ward": "chicago_ward_offices.csv",
    "IUCR": "iucr_codes.csv",
    "Community": "chicago_city_community.csv",
    "Police Beat": "chicago_police_beat_info.csv"
}

for dimension_name, filename in dimension_files.items():
    file_path = RAW_DIR / filename

    print("\n" + "=" * 60)
    print(dimension_name)
    print("=" * 60)

    df = pd.read_csv(file_path, low_memory=False)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(5))


District
Shape: (22, 15)
Columns: ['DISTRICT_CODE', 'DISTRICT_NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'WEBSITE', 'PHONE', 'FAX', 'TTY', 'X_COORDINATE', 'Y_COORDINATE', 'LATITUDE', 'LONGITUDE', 'LOCATION']


,DISTRICT_CODE,DISTRICT_NAME,ADDRESS,CITY,STATE,ZIP,WEBSITE,PHONE,FAX,TTY,X_COORDINATE,Y_COORDINATE,LATITUDE,LONGITUDE,LOCATION
0,1,Central,1718 S State St,Chicago,IL,60616,http://home.chicagopolice.org/community/distri...,312-745-4290,312-745-3694,312-745-3693,1176569.052,1891771.704,41.858373,-87.627356,"(41.8583725929, -87.627356171)"
1,2,Wentworth,5101 S Wentworth Ave,Chicago,IL,60609,http://home.chicagopolice.org/community/distri...,312-747-8366,312-747-5396,312-747-6656,1175864.837,1871153.753,41.801811,-87.630560,"(41.8018110912, -87.6305601801)"
2,3,Grand Crossing,7040 S Cottage Grove Ave,Chicago,IL,60637,http://home.chicagopolice.org/community/distri...,312-747-8201,312-747-5479,312-747-9168,1182739.183,1858317.732,41.766431,-87.605748,"(41.7664308925, -87.6057478606)"
3,4,South Chicago,2255 E 103rd St,Chicago,IL,60617,http://home.chicagopolice.org/community/distri...,312-747-7581,312-747-5276,312-747-9169,1193131.299,1837090.265,41.707933,-87.568349,"(41.7079332906, -87.5683491228)"
4,5,Calumet,727 E 111th St,Chicago,IL,60628,http://home.chicagopolice.org/community/distri...,312-747-8210,312-747-5935,312-747-9170,1183305.427,1831462.313,41.692723,-87.604506,"(41.6927233639, -87.6045058667)"



Ward
Shape: (58, 20)
Columns: ['ward', 'alderman', 'address', 'city', 'state', 'zipcode', 'ward_phone', 'ward_fax', 'email', 'website', 'location_zip', 'location_city', 'location', 'location_address', 'location_state', 'city_hall_address', 'city_hall_city', 'city_hall_state', 'city_hall_zipcode', 'city_hall_phone']


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
0,42,"Reilly, Brendan",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"121 North LaSalle Street, Room 200,Office 6",Chicago,IL,60602.0,(312) 744-3062
1,34,"Austin, Carrie M.",507 West 111th Street,Chicago,IL,60628.0,(773) 928-6961,(773) 928-8562,Ward34@cityofchicago.org,http://www.34thward.org,NaN,NaN,POINT (-87.635452 41.692436),NaN,NaN,"121 North LaSalle Street, Room 200, Office 21",Chicago,IL,60602.0,(312) 744-6820 / 6829
2,46,"Cappleman, James",4544 North Broadway,Chicago,IL,60640.0,(773) 878-4646,(773) 878-4920,Ward46@cityofchicago.org,http://www.James46.org/,NaN,NaN,POINT (-87.656942 41.964594),NaN,NaN,"121 North LaSalle Street, Room 200, Office 10",Chicago,IL,60602.0,(312) 744-6831 / 5694
3,3,"Dowell, Pat",5046 South State Street,Chicago,IL,60609.0,(773) 373-9273,(773) 373-6852,Ward03@cityofchicago.org,http://www.ward03chicago.com,NaN,NaN,POINT (-87.625955 41.802977),NaN,NaN,"121 North LaSalle Street, Room 200, Office 3",Chicago,IL,60602.0,(312) 744-8734 / 6712
4,38,"Sposato, Nicholas",3821 North Harlem Avenue,Chicago,IL,60634.0,(773) 283-3838,(773) 283-3235,Ward38@cityofchicago.org,http://aldermansposato.com/,NaN,NaN,POINT (-87.807131 41.949577),NaN,NaN,"121 North LaSalle Street, Room 200, Office 2",Chicago,IL,60602.0,(312) 744-6857



IUCR
Shape: (410, 4)
Columns: ['IUCR', 'PRIMARY_TYPE ', 'DESCRIPTION', 'INDEX CODE']


,IUCR,PRIMARY_TYPE,DESCRIPTION,INDEX CODE
0,110,HOMICIDE,FIRST DEGREE MURDER,I
1,130,HOMICIDE,SECOND DEGREE MURDER,I
2,141,HOMICIDE,INVOLUNTARY MANSLAUGHTER,N
3,142,HOMICIDE,RECKLESS HOMICIDE,N
4,261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,I



Community
Shape: (77, 7)
Columns: ['community_code', 'community_name', 'population', 'area_sqmile', 'area_sqkm', 'density_per_sqmi', 'density_per_sqkm']


,community_code,community_name,population,area_sqmile,area_sqkm,density_per_sqmi,density_per_sqkm
0,1,Rogers Park,"54,388",1.84,4.77,"29,558.70","11,412.61"
1,2,West Ridge,"78,227",3.53,9.14,"22,160.62","8,556.22"
2,3,Uptown,"57,331",2.32,6.01,"24,711.64","9,541.16"
3,4,Lincoln Square,"41,673",2.56,6.63,"16,278.52","6,285.13"
4,5,North Center,"35,408",2.05,5.31,"17,272.20","6,668.79"



Police Beat
Shape: (277, 4)
Columns: ['DISTRICT', 'SECTOR', 'BEAT', 'BEAT_NUM']


,DISTRICT,SECTOR,BEAT,BEAT_NUM
0,17,1,1,1713
1,31,0,0,3100
2,16,5,5,1651
3,19,1,1,1914
4,19,1,1,1915


In [16]:
ward_file = RAW_DIR / "chicago_ward_offices.csv"

ward_df = pd.read_csv(ward_file)

print("Total records:", len(ward_df))
print("Unique ward numbers:", ward_df["ward"].nunique())

duplicates = ward_df[
    ward_df.duplicated(subset=["ward"], keep=False)
].sort_values("ward")

print("\nDuplicate ward records:")
display(duplicates)

Total records: 58
Unique ward numbers: 50

Duplicate ward records:


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
36,2,"Hopkins, Brian",1400 North Ashland Avenue,Chicago,IL,60622.0,(312) 643-2299,(773) 270-4688,Ward02@cityofchicago.org,https://www.aldermanhopkins.com/,NaN,NaN,POINT (-87.667668 41.907081),NaN,NaN,"121 North LaSalle Street, Room 200, Office 8",Chicago,IL,60602.0,(312) 744-6836
51,2,"Hopkins, Brian","121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6836 / 9867/6213,NaN,office@aldermanhopkins.com,https://www.aldermanhopkins.com/,NaN,NaN,POINT (-87.63227535353653 41.8837645981034),NaN,NaN,NaN,NaN,NaN,NaN,NaN
50,6,"Hall, William E.",8541 South State Street,Chicago,IL,60619.0,(773) 241-3100,NaN,Ward06@cityofchicago.org,NaN,NaN,NaN,POINT (-87.62405576399621 41.73878805068228),NaN,NaN,121 N. LaSalle St. Room 300,Chicago,IL,60602.0,(312) 744-6868
6,6,"Sawyer, Roderick T.",700 East 79th Street,Chicago,IL,60619.0,(773) 635-0006,(773) 891-5679,Ward06@cityofchicago.org,http://www.6ward.com,NaN,NaN,POINT (-87.607315 41.75124),NaN,NaN,"121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6868
23,20,"Taylor, Jeanette B.",5707 South Wentworth Avenue,Chicago,IL,60621.0,(773) 966-5336,NaN,Jeanette.Taylor@cityofchicago.org,NaN,NaN,NaN,POINT (-87.630388 41.7905),NaN,NaN,"121 North LaSalle Street, Room 300, Office 40",Chicago,IL,60602.0,(312) 744-6840
55,20,"Taylor, Jeanette B.","5401 South Wentworth Avenue, Suite 19E",Chicago,IL,60621.0,(773) 966-5336,NaN,Ward20@cityofchicago.org,NaN,NaN,NaN,POINT (-87.63040492438661 41.796234875219376),NaN,NaN,"121 North LaSalle Street, Room 300",Chicago,IL,60602.0,(312) 744-6840/5688/7738
57,27,"Burnett, Walter R.","4 North Western Avenue, Unit 1C",Chicago,IL,60618.0,(312) 432-1995,(312) 432-1049,Ward27@cityofchicago.org,http://aldermanburnett.com/,NaN,NaN,POINT (-87.68658173977015 41.88126893284509),NaN,NaN,"121 North LaSalle Street, Room 300, Office 38",Chicago,IL,60602.0,(312) 744-6124/6125
47,27,"Burnett, Jr., Walter",4 North Western Avenue,Chicago,IL,60612.0,(312) 432-1995,(312) 432-1049,Ward27@cityofchicago.org,http://aldermanburnett.com/,NaN,NaN,POINT (-87.686515 41.881257),NaN,NaN,"121 North LaSalle Street, Room 300, Office 38",Chicago,IL,60602.0,(312) 744-6124 / 6125
52,30,"Cruz, Ruth",5418 W. Belmont Ave,Chicago,IL,60641.0,(773) 628-7874,NaN,Ward30@cityofchicago.org,NaN,NaN,NaN,POINT (-87.76236826862518 41.93873912421377),NaN,NaN,121 N. LaSalle St. Room 200,Chicago,IL,60602.0,(312) 744-3304/3305
21,30,"Reboyras, Ariel",3559 North Milwaukee Avenue,Chicago,IL,60641.0,(773) 794-3095,(773) 794-8576,Ward30@cityofchicago.org,https://www.30thwardchicago.com/,NaN,NaN,POINT (-87.733966 41.945277),NaN,NaN,"121 North LaSalle Street, Room 200, Office 17",Chicago,IL,60602.0,(312) 744-3304 / 3305


In [17]:
# Load the ward reference file
ward_df = pd.read_csv(
    RAW_DIR / "chicago_ward_offices.csv"
)

# Remove leading/trailing spaces from column names
ward_df.columns = ward_df.columns.str.strip()

# Count missing values in each row
ward_df["completeness"] = ward_df.notna().sum(axis=1)

# Sort by ward number and completeness
ward_df = ward_df.sort_values(
    by=["ward", "completeness"],
    ascending=[True, False]
)

# Keep the most complete record for each ward
ward_cleaned = ward_df.drop_duplicates(
    subset=["ward"],
    keep="first"
).copy()

# Remove temporary column
ward_cleaned = ward_cleaned.drop(
    columns=["completeness"]
)

print("Original records:", len(pd.read_csv(
    RAW_DIR / "chicago_ward_offices.csv"
)))

print("Cleaned records:", len(ward_cleaned))
print("Unique ward numbers:", ward_cleaned["ward"].nunique())

display(ward_cleaned.head())

Original records: 58
Cleaned records: 50
Unique ward numbers: 50


,ward,alderman,address,city,state,zipcode,ward_phone,ward_fax,email,website,location_zip,location_city,location,location_address,location_state,city_hall_address,city_hall_city,city_hall_state,city_hall_zipcode,city_hall_phone
35,1,"La Spata, Daniel",1958 North Milwaukee Avenue,Chicago,IL,60647.0,(872) 206-2685,(312) 448-8829,info@the1stward.com,https://www.the1stward.com/,NaN,NaN,POINT (-87.68799 41.917215),NaN,NaN,"121 North LaSalle Street, Room 200, Office 13",Chicago,IL,60602.0,(312) 744-3063
36,2,"Hopkins, Brian",1400 North Ashland Avenue,Chicago,IL,60622.0,(312) 643-2299,(773) 270-4688,Ward02@cityofchicago.org,https://www.aldermanhopkins.com/,NaN,NaN,POINT (-87.667668 41.907081),NaN,NaN,"121 North LaSalle Street, Room 200, Office 8",Chicago,IL,60602.0,(312) 744-6836
3,3,"Dowell, Pat",5046 South State Street,Chicago,IL,60609.0,(773) 373-9273,(773) 373-6852,Ward03@cityofchicago.org,http://www.ward03chicago.com,NaN,NaN,POINT (-87.625955 41.802977),NaN,NaN,"121 North LaSalle Street, Room 200, Office 3",Chicago,IL,60602.0,(312) 744-8734 / 6712
24,4,"King, Sophia D.",435 East 35th Street,Chicago,IL,60616.0,(773) 536-8103,773-536-7296,Ward04@cityofchicago.org,http://www.aldsophiaking.com,NaN,NaN,POINT (-87.61541 41.831135),NaN,NaN,"121 North LaSalle Street, Room 300, Office 29",Chicago,IL,60602.0,(312) 744-2690
8,5,"Hairston, Leslie A.",2325 East 71st Street,Chicago,IL,60649.0,(773) 324-5555,(773) 324-1585,Ward05@cityofchicago.org,http://leslieahairston.com/,NaN,NaN,POINT (-87.567641 41.766147),NaN,NaN,"121 North LaSalle Street, Room 300, Office 26",Chicago,IL,60602.0,(312) 744-6832 / 3195


In [20]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve() / "app"))

from database import get_connection

connection = get_connection()

for table in [
    "dim_district",
    "dim_ward",
    "dim_iucr",
    "dim_community",
    "dim_police_beat"
]:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {table}",
        connection
    )
    print(table, ":", count.iloc[0, 0])

connection.close()

dim_district : 22
dim_ward : 0
dim_iucr : 0
dim_community : 0
dim_police_beat : 0


In [21]:
beat_file = RAW_DIR / "chicago_police_beat_info.csv"

beat_df = pd.read_csv(beat_file)

print("Total records:", len(beat_df))
print("Unique beat numbers:", beat_df["BEAT_NUM"].nunique())

duplicate_beats = beat_df[
    beat_df.duplicated(subset=["BEAT_NUM"], keep=False)
].sort_values("BEAT_NUM")

print("\nDuplicate police beat records:")
display(duplicate_beats)

Total records: 277
Unique beat numbers: 275

Duplicate police beat records:


,DISTRICT,SECTOR,BEAT,BEAT_NUM
1,31,0,0,3100
55,31,0,0,3100
263,31,0,0,3100


In [22]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("..").resolve() / "app"))

from database import get_connection

connection = get_connection()

tables = [
    "crimes",
    "dim_district",
    "dim_ward",
    "dim_iucr",
    "dim_community",
    "dim_police_beat"
]

print("DATABASE VERIFICATION")
print("-" * 40)

for table in tables:
    result = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {table}",
        connection
    )

    print(f"{table}: {result.iloc[0, 0]} records")

connection.close()

DATABASE VERIFICATION
----------------------------------------
crimes: 498 records
dim_district: 22 records
dim_ward: 50 records
dim_iucr: 410 records
dim_community: 77 records
dim_police_beat: 275 records
